# 00 — Setup & Raw Landing (Unity Catalog Volume)

**Purpose:** Create the Unity Catalog namespace + a **Volume**, then land the Kaggle
*E-Commerce Customer Behavior* CSV (uploaded into that Volume) as a clean raw table. The dataset
is small (~350 rows) and contains real defects — notably missing `Satisfaction Level` values —
so the Silver quality pre-checks catch genuine problems.

### Dataset
- **Kaggle:** [E-Commerce Customer Behavior](https://www.kaggle.com/datasets/uom190346a/e-commerce-customer-behavior-dataset)
  — ~350 rows, one customer per row. Tiny, so it runs instantly on **Databricks Free Edition**.

### What This Notebook Does
1. Creates the catalog schemas (`raw`, `bronze`, `silver`, `gold`) and a **UC Volume** for the upload.
2. Reads the CSV you dropped into the Volume, normalises column names, and casts types.
3. Writes `workspace.raw.customers_raw` with **automatic Liquid Clustering** (`CLUSTER BY AUTO`).

### Source & Target
| Direction | Location |
|-----------|----------|
| Source (CSV in UC Volume) | `/Volumes/workspace/raw/landing/` |
| Target (raw staging table) | `workspace.raw.customers_raw` |

> **Prerequisites — upload to Unity Catalog first:**
> 1. **Run the Configuration cell below once** to create the `workspace.raw.landing` Volume.
> 2. In **Catalog → workspace → raw → Volumes → landing**, click **Upload to this volume** and
>    add the Kaggle CSV (any filename — the notebook reads every CSV in the folder).
> 3. Run the rest of the notebook.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG      = 'workspace'                          # Free Edition default catalog
VOLUME       = f'{CATALOG}.raw.landing'             # UC Volume that holds the uploaded CSV
LANDING_PATH = f'/Volumes/{CATALOG}/raw/landing/'   # read every CSV dropped here
RAW_TABLE    = f'{CATALOG}.raw.customers_raw'

# Create schemas + the landing Volume (run this once before uploading the CSV)
for schema in ['raw', 'bronze', 'silver', 'gold']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {VOLUME}')

# Idempotency — drop the raw staging table before re-running
spark.sql(f'DROP TABLE IF EXISTS {RAW_TABLE}')

print(f'Upload the Kaggle CSV here: {LANDING_PATH}')
print(f'Target: {RAW_TABLE}')

In [0]:
from pyspark.sql import functions as F

---

### Step 1 — Read the CSV from the Volume

Reads every CSV file in the landing folder (so the exact filename doesn't matter). `header=true`
picks up the column names; values are cast explicitly in Step 2.

In [0]:
src_df = (
    spark.read
    .option('header', 'true')
    .csv(LANDING_PATH)
)

print(f'Source rows: {src_df.count()}')
print(f'Source columns: {src_df.columns}')
display(src_df.limit(5))

---

### Step 2 — Normalise column names and cast types

The Kaggle CSV uses spaced column names (`Customer ID`, `Membership Type`, ...). We strip spaces,
then select a clean, typed schema. Missing `Satisfaction Level` values are preserved as `NULL`
for the Silver layer to catch.

In [0]:
# Strip spaces from column names (e.g. 'Customer ID' -> 'Customer_ID')
renamed_df = src_df
for col_name in src_df.columns:
    renamed_df = renamed_df.withColumnRenamed(col_name, col_name.strip().replace(' ', '_'))

customers_df = (
    renamed_df
    .select(
        F.col('Customer_ID').cast('int').alias('customer_id'),
        F.col('Gender').alias('gender'),
        F.col('Age').cast('int').alias('age'),
        F.col('City').alias('city'),
        F.col('Membership_Type').alias('membership_type'),
        F.col('Total_Spend').cast('double').alias('total_spend'),
        F.col('Items_Purchased').cast('int').alias('items_purchased'),
        F.col('Average_Rating').cast('double').alias('avg_rating'),
        F.col('Discount_Applied').cast('boolean').alias('discount_applied'),
        F.col('Days_Since_Last_Purchase').cast('int').alias('days_since_last_purchase'),
        F.col('Satisfaction_Level').alias('satisfaction_level'),
    )
)

print(f'Mapped rows: {customers_df.count()}')

In [0]:
# Land the raw staging table, then enable automatic Liquid Clustering
(customers_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(RAW_TABLE))

spark.sql(f'ALTER TABLE {RAW_TABLE} CLUSTER BY AUTO')   # Databricks picks + evolves keys
print(f'Wrote (CLUSTER BY AUTO): {RAW_TABLE}')

---

### Validation — the defects are real

In [0]:
%sql
SELECT
    COUNT(*)                                                       AS total_rows,
    SUM(CASE WHEN satisfaction_level IS NULL THEN 1 ELSE 0 END)    AS missing_satisfaction,
    SUM(CASE WHEN total_spend <= 0           THEN 1 ELSE 0 END)    AS non_positive_spend,
    SUM(CASE WHEN avg_rating NOT BETWEEN 1 AND 5 THEN 1 ELSE 0 END) AS rating_out_of_range
FROM workspace.raw.customers_raw

In [0]:
print(f'Raw row count: {spark.read.table(RAW_TABLE).count()}')
spark.read.table(RAW_TABLE).printSchema()